# Excercise 5
## NLP with Keras

Use keras framework to solve the below exercises.


In [ ]:
import numpy as np
import keras
import pandas as pd
import matplotlib.pyplot as plt

## 5.1 Predict rating of a movie using Keras

**Exercise:** Use keras framework to predict rating.

In [2]:
dataTraining = pd.read_csv('https://github.com/sergiomora03/AdvancedTopicsAnalytics/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)

In [3]:
dataTraining

,year,title,plot,genres,rating
3107,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0
900,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6
6724,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2
4704,1954,Executive Suite,"in a friday afternoon in new york , the presi...",['Drama'],7.4
2582,1990,Narrow Margin,"in los angeles , the editor of a publishing h...","['Action', 'Crime', 'Thriller']",6.6
...,...,...,...,...,...
8417,2010,Our Family Wedding,""" our marriage , their wedding . "" it ' s l...","['Comedy', 'Romance']",4.9
1592,1984,Conan the Destroyer,"the wandering barbarian , conan , alongside ...","['Action', 'Adventure', 'Fantasy']",5.8
1723,1955,Kismet,"like a tale spun by scheherazade , kismet fol...","['Adventure', 'Musical', 'Fantasy', 'Comedy', ...",6.4
7605,1982,The Secret of NIMH,"mrs . brisby , a widowed mouse , lives in a...","['Animation', 'Adventure', 'Drama', 'Family', ...",7.6


In [4]:

plots = dataTraining['plot']
y = (dataTraining['rating'] >= dataTraining['rating'].mean()).astype(int)

In [5]:
plots

3107    most is the story of a single father who takes...
900     a serial killer decides to teach the secrets o...
6724    in sweden ,  a female blackmailer with a disfi...
4704    in a friday afternoon in new york ,  the presi...
2582    in los angeles ,  the editor of a publishing h...
                              ...                        
8417    " our marriage ,  their wedding .  "  it ' s l...
1592    the wandering barbarian ,  conan ,  alongside ...
1723    like a tale spun by scheherazade ,  kismet fol...
7605    mrs .  brisby ,  a widowed mouse ,  lives in a...
215     tinker bell journey far north of never land to...
Name: plot, Length: 7895, dtype: object

In [6]:
y

3107    1
900     0
6724    1
4704    1
2582    1
       ..
8417    0
1592    0
1723    0
7605    1
215     1
Name: rating, Length: 7895, dtype: int32

## Data Precosessing

- Remove stopwords
- Lowercase
- split the text in words
- pad_sequences

In [7]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from nltk import download
from sklearn.model_selection import train_test_split
download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USUARIO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
# --- Preprocessing functions ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation/numbers
    words = text.split()
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

# Apply preprocessing
cleaned_plots = plots.apply(clean_text)

# Tokenization
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(cleaned_plots)
sequences = tokenizer.texts_to_sequences(cleaned_plots)

# Padding sequences
padded_sequences = pad_sequences(sequences, padding='post', maxlen=200)

In [9]:
padded_sequences

array([[  38,  529,   12, ...,    0,    0,    0],
       [ 914,  191,   51, ...,    0,    0,    0],
       [8364,  543, 7247, ...,    0,    0,    0],
       ...,
       [  52,  713,    1, ...,    0,    0,    0],
       [ 370, 9984, 1494, ...,    0,    0,    0],
       [   1, 3097,  383, ...,    0,    0,    0]])

In [10]:
padded_sequences = pad_sequences(sequences, padding='post', maxlen=300)
padded_sequences

array([[  38,  529,   12, ...,    0,    0,    0],
       [ 914,  191,   51, ...,    0,    0,    0],
       [8364,  543, 7247, ...,    0,    0,    0],
       ...,
       [  52,  713,    1, ...,    0,    0,    0],
       [ 370, 9984, 1494, ...,    0,    0,    0],
       [   1, 3097,  383, ...,    0,    0,    0]])

## Build Model

Create a neural network to predict the rating of a movie, calculate the testing set accuracy.

## LSTM Model

In [11]:
# --- Model ---
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=200),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y, test_size=0.2, random_state=42)

# --- Training ---
model.fit(np.array(X_train), np.array(y_train), epochs=5, validation_split=0.2, batch_size=32)

# --- Evaluation ---
loss, accuracy = model.evaluate(np.array(X_test), np.array(y_test))
print(f"Test Accuracy: {accuracy:.2f}")


Epoch 1/5


c:\Users\USUARIO\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


158/158 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.5068 - loss: 0.6934 - val_accuracy: 0.5435 - val_loss: 0.6899
Epoch 2/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 14s 89ms/step - accuracy: 0.5006 - loss: 0.6937 - val_accuracy: 0.5435 - val_loss: 0.6916
Epoch 3/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 13s 84ms/step - accuracy: 0.5206 - loss: 0.6923 - val_accuracy: 0.5435 - val_loss: 0.6912
Epoch 4/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 13s 80ms/step - accuracy: 0.5108 - loss: 0.6935 - val_accuracy: 0.5435 - val_loss: 0.6904
Epoch 5/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - accuracy: 0.5194 - loss: 0.6929 - val_accuracy: 0.5435 - val_loss: 0.6919
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5616 - loss: 0.6914
Test Accuracy: 0.55


En esta parte del ejercicio, se implementa un modelo de red neuronal secuencial con una arquitectura basada en LSTM para predecir si la calificación de una película es superior al promedio. El modelo comienza con una capa de Embedding que transforma las palabras de las sinopsis en vectores numéricos de 64 dimensiones. Esta representación densa permite que el modelo capte relaciones semánticas entre palabras. A continuación, se incorpora una capa LSTM con 64 unidades, la cual es útil para capturar dependencias a largo plazo en secuencias de texto, como aquellas que pueden aparecer en las descripciones de películas. La capa LSTM se configura con return_sequences=False, lo que indica que solo se conservará la última salida de la secuencia, adecuada para tareas de clasificación.

Luego, se añade una capa Dropout con una tasa del 30% para reducir el riesgo de sobreajuste durante el entrenamiento. Seguidamente, una capa densa (Dense) con 32 neuronas y función de activación ReLU permite que el modelo aprenda relaciones no lineales más complejas. Esta capa también está acompañada por una segunda capa de Dropout. Finalmente, la capa de salida con activación sigmoid devuelve un valor entre 0 y 1, ideal para clasificar si la película pertenece a la categoría de “calificación alta” o “calificación baja”.

El modelo se compila usando la función de pérdida binary_crossentropy —adecuada para clasificación binaria— y el optimizador adam, que es eficiente y robusto para este tipo de tareas. Los datos se dividen en entrenamiento y prueba, con el 80% de las observaciones destinadas al entrenamiento y el 20% a la evaluación. Durante el entrenamiento, se reserva además un 20% del conjunto de entrenamiento como validación. El modelo se entrena durante cinco épocas con lotes de 32 ejemplos por iteración. Finalmente, se evalúa el rendimiento del modelo sobre el conjunto de prueba, y se imprime su precisión como métrica principal.

## LSTM & GRU Models 

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y, test_size=0.2, random_state=42)

# Convert to numpy arrays
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

# --- 1. Model with LSTM ---
model_lstm = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=200),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train LSTM model
print("Training LSTM model...")
model_lstm.fit(X_train, y_train, epochs=5, validation_split=0.2, batch_size=32)

# Evaluate
loss_lstm, acc_lstm = model_lstm.evaluate(X_test, y_test)
print(f"\n LSTM Model - Test Accuracy: {acc_lstm:.4f}")

# --- 2. Model with GRU ---
model_gru = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=200),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_gru.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train GRU model
print("\nTraining GRU model...")
model_gru.fit(X_train, y_train, epochs=5, validation_split=0.2, batch_size=32)

# Evaluate
loss_gru, acc_gru = model_gru.evaluate(X_test, y_test)
print(f"\n GRU Model - Test Accuracy: {acc_gru:.4f}")
import matplotlib.pyplot as plt

Training LSTM model...
Epoch 1/5


c:\Users\USUARIO\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


158/158 ━━━━━━━━━━━━━━━━━━━━ 25s 145ms/step - accuracy: 0.5146 - loss: 0.6930 - val_accuracy: 0.4573 - val_loss: 0.6932
Epoch 2/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 21s 134ms/step - accuracy: 0.4953 - loss: 0.6941 - val_accuracy: 0.5435 - val_loss: 0.6911
Epoch 3/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 20s 127ms/step - accuracy: 0.5179 - loss: 0.6925 - val_accuracy: 0.4581 - val_loss: 0.6932
Epoch 4/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 20s 128ms/step - accuracy: 0.5211 - loss: 0.6927 - val_accuracy: 0.5435 - val_loss: 0.6900
Epoch 5/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 20s 126ms/step - accuracy: 0.5209 - loss: 0.6923 - val_accuracy: 0.5435 - val_loss: 0.6915
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.5616 - loss: 0.6907

 LSTM Model - Test Accuracy: 0.5478

Training GRU model...
Epoch 1/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 16s 90ms/step - accuracy: 0.5021 - loss: 0.6934 - val_accuracy: 0.5435 - val_loss: 0.6898
Epoch 2/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.5030 - loss: 0.6940 - val_accu

En este ejercicio, se construyen dos modelos de redes neuronales —uno con LSTM y otro con GRU— para predecir si una película tiene una calificación por encima del promedio, utilizando como insumo su sinopsis. Primero, se dividen los datos en conjunto de entrenamiento (80%) y prueba (20%) mediante la función train_test_split, y se convierten a arreglos NumPy para asegurar compatibilidad con Keras. Luego, se crea el primer modelo utilizando la clase Sequential de Keras. Este modelo incluye una capa de Embedding que convierte las palabras en vectores numéricos de 128 dimensiones, seguida por una capa LSTM con 64 unidades que permite capturar relaciones temporales en el texto. Para evitar el sobreajuste, se añaden capas de Dropout con una tasa del 30%. Posteriormente, se agrega una capa Dense con 64 neuronas y activación ReLU, y una capa de salida con activación sigmoid, ideal para clasificación binaria. El modelo se compila con la función de pérdida binary_crossentropy y el optimizador adam, y se entrena durante 5 épocas con un tamaño de lote de 32, reservando un 20% del conjunto de entrenamiento para validación.

Después del entrenamiento, se evalúa el modelo sobre el conjunto de prueba y se imprime su precisión. De forma paralela, se construye un segundo modelo que replica la misma arquitectura, pero reemplazando la capa LSTM por una capa GRU. Las GRU son una alternativa más eficiente a las LSTM, con un menor costo computacional y, en algunos casos, resultados comparables. Este segundo modelo también se compila, entrena y evalúa de la misma manera, permitiendo comparar el desempeño de ambas arquitecturas en la tarea de clasificación. Finalmente, se observa la precisión de ambos modelos en el conjunto de prueba, lo que permite identificar cuál de las dos arquitecturas es más adecuada para este problema específico.

## Conclusiones

Al evaluar el desempeño de los modelos construidos con LSTM y GRU, se observa que la precisión alcanzada en el conjunto de prueba se mantiene relativamente constante entre las distintas configuraciones. El modelo con GRU obtuvo una precisión de aproximadamente 54.78%, mientras que el modelo con LSTM logró un 54.78% al usar 128 unidades y una leve mejora hasta el 55% con 64 unidades. Estos resultados sugieren que, aunque ambos modelos logran superar el azar (50%), aún no están capturando completamente los patrones del texto que determinan la calificación de una película. Es posible que la arquitectura actual no sea suficientemente compleja para el problema, o que la información contenida en las sinopsis no sea del todo representativa para predecir el puntaje.